# Repulsive Curves And Proteins

This notebook is one chapter of the runnable `KnottedGraph` user guide.  It is
generated into `User_guide/07_repulsive_curves_and_proteins.ipynb` so users can open the specific workflow
they need without navigating one very large notebook.

- self-contained after the shared setup cells


## 0. Setup, Preflight, And Shared Plot Style

The whole notebook uses the same visual convention:

- blue: surfaces, skeleton points, and graph edges;
- red: graph vertices;
- black axes;
- paper notation: `Upsilon(G; Y)`.

The helper functions in this section remove repeated plotting boilerplate from
the rest of the notebook.  This is also the library-level pattern worth
promoting later into public visualization helpers.


In [1]:
from pathlib import Path
import sys
import importlib.util
import os
import tempfile

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

DOC_ROOT = PROJECT_ROOT / "doc"
SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "knottedgraph-mpl"))

print("project paths configured")
for package in ["numpy", "networkx", "sympy", "plotly", "matplotlib", "pyvista"]:
    print(f"{package:10s} = {importlib.util.find_spec(package) is not None}")


project paths configured
numpy      = True
networkx   = True
sympy      = True
plotly     = True
matplotlib = True
pyvista    = True


In [2]:
import math
import time

import networkx as nx
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
import sympy as sp
from IPython.display import Math, display

from knotted_graph.projection import (
    compute_yamada_polynomial,
    sample_projections,
    select_projection,
)
from knotted_graph.visualization import plot_3D_graph_plotly

BLUE = "#1f77b4"
RED = "#d62728"
CAMERA = dict(eye=dict(x=1.45, y=1.55, z=1.18))
pio.renderers.default = "notebook_connected"
Y = sp.Symbol("Y")
kx, ky, kz = sp.symbols("k_x k_y k_z", real=True)


def axis_style():
    return dict(
        visible=True,
        title="",
        showticklabels=False,
        showbackground=False,
        showgrid=False,
        zeroline=False,
        showline=True,
        linecolor="black",
        linewidth=2,
    )


def apply_kg_layout(fig, *, width=760, height=620):
    fig.update_layout(
        title=None,
        width=width,
        height=height,
        margin=dict(l=0, r=0, t=0, b=0),
        scene=dict(
            xaxis=axis_style(),
            yaxis=axis_style(),
            zaxis=axis_style(),
            aspectmode="data",
            camera=CAMERA,
        ),
    )
    return fig


def plot_surface_polydata(surface, *, opacity=0.58):
    mesh = surface.triangulate()
    faces = mesh.faces.reshape(-1, 4)[:, 1:]
    pts = mesh.points
    fig = go.Figure(
        go.Mesh3d(
            x=pts[:, 0],
            y=pts[:, 1],
            z=pts[:, 2],
            i=faces[:, 0],
            j=faces[:, 1],
            k=faces[:, 2],
            color=BLUE,
            opacity=opacity,
        )
    )
    return apply_kg_layout(fig)


def plot_points_3d(points, *, size=3):
    points = np.asarray(points)
    fig = go.Figure(
        go.Scatter3d(
            x=points[:, 0],
            y=points[:, 1],
            z=points[:, 2],
            mode="markers",
            marker=dict(size=size, color=BLUE),
        )
    )
    return apply_kg_layout(fig)


def plot_graph_kg(graph):
    return apply_kg_layout(plot_3D_graph_plotly(graph))


def print_upsilon(label, expr):
    print(f"Upsilon({label}; Y) = {sp.expand(expr)}")


def display_bloch_vector(label, components):
    display(Math(label + r"=" + sp.latex(sp.Matrix(components))))


print("shared plotting and notation helpers ready")


shared plotting and notation helpers ready


## 7. Repulsive-Curve Workflow For Complicated Embeddings

The public package can build protein-derived theta graphs and evaluate small
examples directly.  The full repulsive optimization requires the external
Repulsor driver, so the optimization call is shown as reference code below,
while the initial protein graph and invariant are executable.


# To be filled by Kehan

This section is marked for Kehan because it covers protein-derived theta graphs
and repulsive-curve layouts.  The executable examples below are placeholders
showing the current public calls and expected outputs.


In [3]:
from pathlib import Path

from knotted_graph.layout.repulsive import (
    available_samples,
    build_protein_example,
    curve_network_to_multigraph,
)
from knotted_graph.layout.repulsive.protein_examples import set_special_node_distance

print("available protein examples =", available_samples())

protein_networks = {}
protein_graphs = {}
for sample in available_samples():
    network = build_protein_example(
        sample,
        pdb_cache=PROJECT_ROOT / "pdb-cache",
        total_arc_points=42 if sample == "1aoc" else 54,
    )
    if sample == "1aoc":
        set_special_node_distance(network, target_distance=9.0)
    protein_networks[sample] = network
    protein_graphs[sample] = curve_network_to_multigraph(network)
    print(sample)
    print("  name =", network.name)
    print("  nodes =", network.node_order)
    print("  arcs =", network.arc_order)
    print("  nodes_edges =", (protein_graphs[sample].number_of_nodes(), protein_graphs[sample].number_of_edges()))

network = protein_networks["1aoc"]
protein_graph = protein_graphs["1aoc"]


available protein examples = ('1aoc', '3ulk', '5osq')
1aoc
  name = 1AOC theta_31
  nodes = ('C140', 'C134')
  arcs = ('arc1', 'arc2', 'arc3')
  nodes_edges = (2, 3)


3ulk
  name = 3ULK theta_41
  nodes = ('D217', 'E393')
  arcs = ('arc1_closure', 'arc2_backbone', 'arc3_mg_bridge')
  nodes_edges = (2, 3)
5osq
  name = 5OSQ theta
  nodes = ('D437', 'C469')
  arcs = ('arc1_ca_closure', 'arc2_cys_bridge', 'arc3_backbone')
  nodes_edges = (2, 3)


In [4]:
fig = plot_graph_kg(protein_graph)
fig.show()


In [5]:
fig = plot_graph_kg(protein_graphs["3ulk"])
fig.show()


In [6]:
fig = plot_graph_kg(protein_graphs["5osq"])
fig.show()


In [7]:
protein_projection = select_projection(protein_graph, num_rotation_samples=12)
protein_result = compute_yamada_polynomial(
    protein_graph,
    Y,
    rotation_angles=protein_projection.rotation_angles,
    return_result=True,
    n_jobs=1,
)
print(f"selected_crossings = {protein_projection.num_crossings}")
print(f"pd_code = {protein_projection.pd_code}")
print_upsilon("G_1AOC", protein_result.polynomial)


selected_crossings = 5
pd_code = V[11,0,6];V[12,10,5];X[8,3,7,4];X[12,9,11,10];X[1,8,0,9];X[4,7,5,6]
Upsilon(G_1AOC; Y) = -Y**12 - Y**11 - Y**10 - Y**9 - Y**8 - Y**6 - Y**4 + 1


Reference code for the optional external-driver layout step:

```python
from knotted_graph.layout.repulsive import SolverOptions, relax_spatial_graph

layout = relax_spatial_graph(
    protein_graph,
    workspace="protein-layout",
    solver_options=SolverOptions(steps=100, max_time=20, threads=1),
    save_steps=True,
    keep_workspace=True,
    verify_topology=True,
)

relaxed_graph = layout.graph
```

This should become a fully executable paper cell once the external driver is
installed and the layout workspace is standardized for distribution.


The before/after panel records the expected visual purpose of the repulsive
stage: simplify the geometric representative while preserving the graph
incidence and, after successful topology verification, the invariant.

No static before/after image is displayed here.  Once the external Repulsor
driver is available in the environment, run the reference code above and then
plot `protein_graph` and `relaxed_graph` with `plot_graph_kg(...)` in adjacent
Plotly scenes.
